### In this notebook I create the national PFAS site map and one map for each state.

### This version is based on Notebook 2, because the map popups need information from the row-level PFAS results. I create the site-level map table directly from the cleaned Notebook 2 file so the popup has detected parameters, total parameters tested, total testing instances, and positive parameters.

In [1]:
import pandas as pd
import numpy as np
import folium
import os
import html

In [2]:
file_path = "G:/My Drive/Projects/Large projects/PFAS in water/"

site_level = pd.read_csv(
    file_path + "saved files/Notebook3/pfas_site_level.csv",
    low_memory=False
)

output_folder = file_path + "saved files/Notebook4/maps/"
os.makedirs(output_folder, exist_ok=True)

site_level["ActivityLocation/LatitudeMeasure"] = pd.to_numeric(
    site_level["ActivityLocation/LatitudeMeasure"],
    errors="coerce"
)

site_level["ActivityLocation/LongitudeMeasure"] = pd.to_numeric(
    site_level["ActivityLocation/LongitudeMeasure"],
    errors="coerce"
)

site_level["max_pfas_ng"] = pd.to_numeric(
    site_level["max_pfas"],
    errors="coerce"
)


In [3]:
# I use one popup function so the national map and state maps show the same information.

def make_pfas_popup(row):

    max_pfas = row["max_pfas_ng"]

    if pd.notna(max_pfas):
        max_pfas_text = f"{max_pfas:,.2f} ng/L"
    else:
        max_pfas_text = "No numeric value"

    popup_html = f"""
    <b>Location:</b> {html.escape(str(row["MonitoringLocationName"]))}<br>
    <b>ID:</b> {html.escape(str(row["MonitoringLocationIdentifier"]))}<br>
    <b>State:</b> {html.escape(str(row["state"]))}<br>
    <b>PFAS detected:</b> {html.escape(str(row["location_pfas_detected"]))}<br>
    <b>Sampling detection category:</b> {html.escape(str(row["detection_sampling_event_category"]))}<br>
    <b>Maximum PFAS:</b> {html.escape(str(max_pfas_text))}<br><br>

    <b>Testing instances:</b> {html.escape(str(row["testing_instances"]))}<br>
    <b>Sampling events:</b> {html.escape(str(row["sampling_events"]))}<br>
    <b>Sampling events with detection:</b> {html.escape(str(row["sampling_events_with_detection"]))}<br>
    <b>% sampling events with detection:</b> {html.escape(str(row["percent_sampling_events_with_detection"]))}%<br><br>

    <b>Parameters tested:</b> {html.escape(str(row["parameters_tested"]))}<br>
    <b>Parameters positive:</b> {html.escape(str(row["parameters_positive"]))}
    """

    return folium.Popup(
        popup_html,
        max_width=400
    )


# I define marker fill colors, but I keep the black border around every point.

def pfas_fill_color(status):
    if status == "Y":
        return "red"
    elif status == "N":
        return "green"
    elif status == "U":
        return "orange"
    else:
        return "white"


# I create the national PFAS map.

m = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="OpenStreetMap"
)

for _, row in site_level.iterrows():

    lat = row["ActivityLocation/LatitudeMeasure"]
    lon = row["ActivityLocation/LongitudeMeasure"]

    if pd.isna(lat) or pd.isna(lon):
        continue

    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="black",
        weight=1,
        fill=True,
        fill_color=pfas_fill_color(row["location_pfas_detected"]),
        fill_opacity=0.9,
        popup=make_pfas_popup(row)
    ).add_to(m)


# I update the legend so it explains both color and sampling category.

legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 310px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding:10px;
">

<b>PFAS Detection Status</b><br><br>

<span style="color:red;">●</span>
PFAS detected at location<br>

<span style="color:green;">●</span>
No PFAS detected<br>


<b>Popup sampling categories</b><br>
PFAS detected in 100% of sampling<br>
PFAS detected in less than 100% of sampling<br>
No PFAS detected

</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)

map_path = output_folder + "pfas_national_site_map.html"

m.save(map_path)

print("Saved:", map_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_national_site_map.html


In [4]:
# I create one popup function so every state map has the same popup information.

def make_pfas_popup(row):

    max_pfas = row["max_pfas"]

    if pd.notna(max_pfas):
        max_pfas_text = f"{max_pfas:,.2f} ng/L"
    else:
        max_pfas_text = "No numeric value"

    popup_html = f"""
    <b>Location:</b> {html.escape(str(row["MonitoringLocationName"]))}<br>
    <b>ID:</b> {html.escape(str(row["MonitoringLocationIdentifier"]))}<br>
    <b>State:</b> {html.escape(str(row["state"]))}<br>
    <b>PFAS detected:</b> {html.escape(str(row["location_pfas_detected"]))}<br>
    <b>Sampling category:</b> {html.escape(str(row["detection_sampling_event_category"]))}<br>
    <b>Maximum PFAS:</b> {html.escape(str(max_pfas_text))}<br><br>

    <b>Testing instances:</b> {html.escape(str(row["testing_instances"]))}<br>
    <b>Sampling events:</b> {html.escape(str(row["sampling_events"]))}<br>
    <b>Sampling events with detection:</b> {html.escape(str(row["sampling_events_with_detection"]))}<br>
    <b>% sampling events with detection:</b> {html.escape(str(row["percent_sampling_events_with_detection"]))}%<br><br>

    <b>Parameters tested:</b> {html.escape(str(row["parameters_tested"]))}<br>
    <b>Parameters positive:</b> {html.escape(str(row["parameters_positive"]))}
    """

    return folium.Popup(
        popup_html,
        max_width=400
    )


# I use fill color for PFAS status, but I keep a black border around all points.

def pfas_fill_color(status):

    if status == "Y":
        return "red"
    elif status == "N":
        return "green"
    elif status == "U":
        return "orange"
    else:
        return "white"


# I create a legend for the state maps.

legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 320px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding:10px;
">

<b>PFAS Detection Status</b><br><br>

<span style="color:red;">●</span> PFAS detected<br>
<span style="color:green;">●</span> No PFAS detected<br>

<b>Popup sampling categories</b><br>
PFAS detected in 100% of sampling<br>
PFAS detected in less than 100% of sampling<br>
No PFAS detected

</div>
"""


# I create one PFAS map for each state.

for state in sorted(site_level["state"].dropna().unique()):

    state_data = site_level.loc[
        site_level["state"] == state
    ].copy()

    state_data = state_data.dropna(
        subset=[
            "ActivityLocation/LatitudeMeasure",
            "ActivityLocation/LongitudeMeasure"
        ]
    )

    if state_data.empty:
        continue

    center_lat = state_data["ActivityLocation/LatitudeMeasure"].mean()
    center_lon = state_data["ActivityLocation/LongitudeMeasure"].mean()

    m_state = folium.Map(
        location=[center_lat, center_lon],
        zoom_start=7,
        tiles="OpenStreetMap"
    )

    for _, row in state_data.iterrows():

        folium.CircleMarker(
            location=[
                row["ActivityLocation/LatitudeMeasure"],
                row["ActivityLocation/LongitudeMeasure"]
            ],
            radius=8,
            color="black",
            weight=1,
            fill=True,
            fill_color=pfas_fill_color(row["location_pfas_detected"]),
            fill_opacity=0.85,
            popup=make_pfas_popup(row)
        ).add_to(m_state)

    m_state.get_root().html.add_child(
        folium.Element(legend_html)
    )

    state_file = (
        output_folder
        + "pfas_map_"
        + str(state).replace(" ", "_").replace("/", "_")
        + ".html"
    )

    m_state.save(state_file)

    print("Saved:", state_file)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_AK.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_AL.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_AR.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_AZ.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_CA.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_CO.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_CT.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_D.C..html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_map_DE.html
Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_

In [6]:
# I load the state boundary file so I can add state outlines to the PFAS map.

import geopandas as gpd

states_gdf = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_500k.zip"
)

# I create the national PFAS map.

m = folium.Map(
    location=[39.5, -98.35],
    zoom_start=4,
    tiles="OpenStreetMap"
)

# I add state boundaries to the national PFAS map.

folium.GeoJson(
    states_gdf,
    name="State boundaries",
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "black",
        "weight": 0.8,
        "fillOpacity": 0
    }
).add_to(m)

# I add PFAS locations to the map.
# I exclude locations classified as uncertain.

for _, row in site_level.iterrows():

    lat = row["ActivityLocation/LatitudeMeasure"]
    lon = row["ActivityLocation/LongitudeMeasure"]
    status = row["location_pfas_detected"]

    if pd.isna(lat) or pd.isna(lon):
        continue

    # I do not map uncertain locations.
    if status == "U":
        continue

    # I assign colors only to detected and not-detected locations.
    if status == "Y":
        fill_color = "red"
    elif status == "N":
        fill_color = "green"
    else:
        continue

    folium.CircleMarker(
        location=[lat, lon],
        radius=8,
        color="black",
        weight=1,
        fill=True,
        fill_color=fill_color,
        fill_opacity=0.9,
        popup=make_pfas_popup(row)
    ).add_to(m)


# I add a legend explaining the PFAS detection categories.

legend_html = """
<div style="
    position: fixed;
    bottom: 50px;
    left: 50px;
    width: 280px;
    background-color: white;
    border: 2px solid black;
    z-index: 9999;
    font-size: 14px;
    padding: 10px;
">
<b>PFAS Detection Status</b><br><br>

<span style="
    display:inline-block;
    width:12px;
    height:12px;
    background:red;
    border:1px solid black;
    border-radius:50%;
"></span>
&nbsp;PFAS detected<br><br>

<span style="
    display:inline-block;
    width:12px;
    height:12px;
    background:green;
    border:1px solid black;
    border-radius:50%;
"></span>
&nbsp;No PFAS detected
</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html)
)


legend_html_2 = """
<div style="
    position: fixed;
    bottom: 50px;
    right: 30px;
    width: 300px;
    background-color: white;
    border: 2px solid black;
    z-index: 9999;
    font-size: 11px;
    padding: 10px;
    line-height: 1.25;
">
<b>Site Metrics</b><br><br>

<b>Testing instances</b><br>
PFAS test results reported.<br><br>

<b>Sampling events</b><br>
Unique sampling dates.<br><br>

<b>Events with detection</b><br>
Sampling dates with PFAS detected.<br><br>

<b>% events with detection</b><br>
Percent of sampling dates with PFAS detected.<br><br>

<b>Parameters tested</b><br>
Unique PFAS compounds analyzed.<br><br>

<b>Parameters positive</b><br>
Unique PFAS compounds detected.<br><br>

<b>Maximum PFAS</b><br>
Highest reported concentration (ng/L).<br><br>

<b>Sampling category</b><br>
100% = detected every sampling event.<br>
&lt;100% = detected in some events.<br>
No PFAS detected = no detections reported.
</div>
"""

m.get_root().html.add_child(
    folium.Element(legend_html_2)
)


# I save the national PFAS map with state borders.

map_path = os.path.join(
    output_folder,
    "pfas_national_site_map_with_borders.html"
)

m.save(map_path)

print("Saved:", map_path)

Saved: G:/My Drive/Projects/Large projects/PFAS in water/saved files/Notebook4/maps/pfas_national_site_map_with_borders.html
